In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# TensorFlow/Keras for LSTM
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Settings
pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Paths
PROCESSED_DATA_DIR = '../../data/processed'
MODELS_DIR = '../../models/dl'
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"TensorFlow version: {tf.__version__}")

# ============================================
# GPU Configuration
# ============================================
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU Available: {gpus}")

if gpus:
    try:
        # Enable memory growth to avoid allocating all GPU memory at once
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

        # Enable mixed precision for faster training on GPU
        from tensorflow.keras import mixed_precision
        mixed_precision.set_global_policy('mixed_float16')

        print(f"\n✓ GPU configured successfully!")
        print(f"✓ Mixed precision enabled (float16)")
        print(f"✓ Using {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")
else:
    print("\n⚠ No GPU detected. Training will use CPU (slower)")
    print("\nTo enable GPU:")
    print("  - Mac with M1/M2/M3: pip install tensorflow-metal")
    print("  - NVIDIA GPU: Install CUDA and cuDNN, then: pip install tensorflow[and-cuda]")

## 1. Load Processed Data

Load the preprocessed train/validation/test splits.

In [2]:
# Load preprocessed data
train_df = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, 'train_split.parquet'))
val_df = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, 'val_split.parquet'))
test_df = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, 'test_split.parquet'))

print(f"Train shape: {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

# Check columns
print(f"\nColumns: {train_df.columns.tolist()}")

Train shape: (14151270, 32)
Validation shape: (3032415, 32)
Test shape: (3032415, 32)

Columns: ['building_id', 'meter', 'timestamp', 'meter_reading', 'site_id', 'primary_use', 'square_feet', 'year_built', 'floor_count', 'air_temperature', 'cloud_coverage', 'dew_temperature', 'precip_depth_1_hr', 'sea_level_pressure', 'wind_direction', 'wind_speed', 'hour', 'day', 'weekday', 'month', 'is_weekend', 'log_square_feet', 'building_age', 'log_meter_reading', 'meter_reading_per_sqft', 'log_meter_per_sqft', 'sqft_per_floor', 'cooling_degree_hours', 'heating_degree_hours', 'meter_reading_lag1', 'hourly_avg_per_building', 'weekend_avg_per_building']


## 2. Prepare Data for LSTM

### 2.1 Feature Selection & Scaling

Select relevant features and scale them for LSTM training.

In [3]:
# Select features for LSTM (exclude timestamp, building_id, and target)
# Focus on temporal, weather, and building features
feature_cols = [
    'hour', 'day', 'weekday', 'month', 'is_weekend',
    'log_square_feet', 'building_age', 'floor_count',
    'air_temperature', 'meter_reading_per_sqft',
    'cooling_degree_hours', 'heating_degree_hours',
    'meter_reading_lag1', 'hourly_avg_per_building',
    'weekend_avg_per_building'
]

# Filter to columns that exist in the dataset
feature_cols = [col for col in feature_cols if col in train_df.columns]
target_col = 'log_meter_reading'

print(f"Using {len(feature_cols)} features for LSTM")
print(f"Features: {feature_cols}")

# Extract features and target
X_train = train_df[feature_cols].values
y_train = train_df[target_col].values

X_val = val_df[feature_cols].values
y_val = val_df[target_col].values

X_test = test_df[feature_cols].values
y_test = test_df[target_col].values

print(f"\nX_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

Using 15 features for LSTM
Features: ['hour', 'day', 'weekday', 'month', 'is_weekend', 'log_square_feet', 'building_age', 'floor_count', 'air_temperature', 'meter_reading_per_sqft', 'cooling_degree_hours', 'heating_degree_hours', 'meter_reading_lag1', 'hourly_avg_per_building', 'weekend_avg_per_building']

X_train shape: (14151270, 15)
y_train shape: (14151270,)

X_train shape: (14151270, 15)
y_train shape: (14151270,)


In [ ]:
# Scale features using MinMaxScaler (LSTM works better with normalized inputs)
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train_clean)
X_val_scaled = scaler_X.transform(X_val_clean)
X_test_scaled = scaler_X.transform(X_test_clean)

y_train_scaled = scaler_y.fit_transform(y_train_clean.reshape(-1, 1)).flatten()
y_val_scaled = scaler_y.transform(y_val_clean.reshape(-1, 1)).flatten()
y_test_scaled = scaler_y.transform(y_test_clean.reshape(-1, 1)).flatten()

print("Data scaled successfully.")
print(f"No NaN in scaled data: {np.isnan(X_train_scaled).sum() == 0}")

Data scaled successfully.


In [ ]:
# Check for NaN in raw data before scaling
print("Checking raw data quality...")
print(f"X_train - NaN: {np.isnan(X_train).sum()}, Inf: {np.isinf(X_train).sum()}")
print(f"y_train - NaN: {np.isnan(y_train).sum()}, Inf: {np.isinf(y_train).sum()}")

# Remove rows with NaN
print("\nRemoving NaN values...")
train_mask = ~(np.isnan(X_train).any(axis=1) | np.isnan(y_train))
val_mask = ~(np.isnan(X_val).any(axis=1) | np.isnan(y_val))
test_mask = ~(np.isnan(X_test).any(axis=1) | np.isnan(y_test))

X_train_clean = X_train[train_mask]
y_train_clean = y_train[train_mask]

X_val_clean = X_val[val_mask]
y_val_clean = y_val[val_mask]

X_test_clean = X_test[test_mask]
y_test_clean = y_test[test_mask]

print(f"\nAfter cleaning:")
print(f"Train: {X_train_clean.shape[0]} samples (removed {X_train.shape[0] - X_train_clean.shape[0]})")
print(f"Val: {X_val_clean.shape[0]} samples (removed {X_val.shape[0] - X_val_clean.shape[0]})")
print(f"Test: {X_test_clean.shape[0]} samples (removed {X_test.shape[0] - X_test_clean.shape[0]})")

### 2.2 Create Sequences for LSTM

LSTM models require sequences of past observations to predict future values.
We'll create sliding windows of `timesteps` observations.

In [5]:
def create_sequences(X, y, timesteps=24):
    """
    Create sequences for LSTM training.
    
    Parameters:
    - X: features array (n_samples, n_features)
    - y: target array (n_samples,)
    - timesteps: number of past timesteps to use as input
    
    Returns:
    - X_seq: sequences (n_samples - timesteps, timesteps, n_features)
    - y_seq: targets (n_samples - timesteps,)
    """
    X_seq, y_seq = [], []
    
    for i in range(len(X) - timesteps):
        X_seq.append(X[i:i + timesteps])
        y_seq.append(y[i + timesteps])
    
    return np.array(X_seq), np.array(y_seq)

# Define timesteps (24 hours = 1 day lookback)
TIMESTEPS = 24

# Create sequences
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, TIMESTEPS)
X_val_seq, y_val_seq = create_sequences(X_val_scaled, y_val_scaled, TIMESTEPS)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, TIMESTEPS)

print(f"\nSequence shapes:")
print(f"X_train_seq: {X_train_seq.shape}  (samples, timesteps, features)")
print(f"y_train_seq: {y_train_seq.shape}")
print(f"X_val_seq: {X_val_seq.shape}")
print(f"y_val_seq: {y_val_seq.shape}")


Sequence shapes:
X_train_seq: (14151246, 24, 15)  (samples, timesteps, features)
y_train_seq: (14151246,)
X_val_seq: (3032391, 24, 15)
y_val_seq: (3032391,)


In [6]:
# Check for NaN/Inf in data
print("Checking data quality...")
print(f"X_train_seq - NaN: {np.isnan(X_train_seq).sum()}, Inf: {np.isinf(X_train_seq).sum()}")
print(f"y_train_seq - NaN: {np.isnan(y_train_seq).sum()}, Inf: {np.isinf(y_train_seq).sum()}")
print(f"X_val_seq - NaN: {np.isnan(X_val_seq).sum()}, Inf: {np.isinf(X_val_seq).sum()}")
print(f"y_val_seq - NaN: {np.isnan(y_val_seq).sum()}, Inf: {np.isinf(y_val_seq).sum()}")

# Check value ranges
print(f"\nX_train_seq range: [{X_train_seq.min():.4f}, {X_train_seq.max():.4f}]")
print(f"y_train_seq range: [{y_train_seq.min():.4f}, {y_train_seq.max():.4f}]")

Checking data quality...
X_train_seq - NaN: 1843545, Inf: 0
X_train_seq - NaN: 1843545, Inf: 0
y_train_seq - NaN: 0, Inf: 0
y_train_seq - NaN: 0, Inf: 0
X_val_seq - NaN: 163944, Inf: 0
y_val_seq - NaN: 0, Inf: 0
X_val_seq - NaN: 163944, Inf: 0
y_val_seq - NaN: 0, Inf: 0

X_train_seq range: [nan, nan]

X_train_seq range: [nan, nan]
y_train_seq range: [0.0000, 1.0000]
y_train_seq range: [0.0000, 1.0000]


## 3. Build LSTM Model

### Architecture:
- Stacked LSTM layers with dropout for regularization
- Dense output layer for regression
- Adam optimizer with MSE loss

In [7]:
def build_lstm_model(input_shape, lstm_units=[128, 64], dropout=0.2, learning_rate=0.0001):
    """
    Build a stacked LSTM model for time-series forecasting.
    
    Parameters:
    - input_shape: (timesteps, n_features)
    - lstm_units: list of units for each LSTM layer
    - dropout: dropout rate for regularization
    - learning_rate: learning rate for Adam optimizer
    
    Returns:
    - Compiled Keras model
    """
    model = Sequential(name='LSTM_Energy_Forecaster')
    
    # First LSTM layer (return sequences for stacking)
    model.add(layers.LSTM(
        units=lstm_units[0],
        return_sequences=True,
        input_shape=input_shape,
        name='lstm_1'
    ))
    model.add(layers.Dropout(dropout, name='dropout_1'))
    
    # Second LSTM layer
    if len(lstm_units) > 1:
        model.add(layers.LSTM(
            units=lstm_units[1],
            return_sequences=False,
            name='lstm_2'
        ))
        model.add(layers.Dropout(dropout, name='dropout_2'))
    
    # Dense output layer
    model.add(layers.Dense(32, activation='relu', name='dense_1'))
    model.add(layers.Dense(1, name='output'))
    
    # Compile model with gradient clipping
    optimizer = keras.optimizers.Adam(
        learning_rate=learning_rate,
        clipnorm=1.0  # Gradient clipping to prevent explosion
    )
    model.compile(
        optimizer=optimizer,
        loss='mse',
        metrics=['mae']
    )
    
    return model

# Build model with lower learning rate
input_shape = (TIMESTEPS, X_train_seq.shape[2])
model = build_lstm_model(input_shape, lstm_units=[128, 64], dropout=0.2, learning_rate=0.0001)

# Print model summary
print(model.summary())

Model: "LSTM_Energy_Forecaster"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 24, 128)        │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 125,249 (489.25 KB)

 Trainable params: 125,249 (489.25 KB)

 Non-trainable params: 0 (0.00 B)

None


## 4. Train LSTM Model

Train with callbacks:
- Early stopping to prevent overfitting
- Model checkpoint to save best weights
- Learning rate reduction on plateau

In [ ]:
# Define callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    filepath=os.path.join(MODELS_DIR, 'lstm_energy_forecaster.keras'),
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# Train model
# Use larger batch size if GPU is available for better throughput
BATCH_SIZE = 512 if len(tf.config.list_physical_devices('GPU')) > 0 else 256
EPOCHS = 50

print(f"\nTraining configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Training samples: {len(X_train_seq):,}")
print(f"  Validation samples: {len(X_val_seq):,}")
print(f"  Device: {'GPU' if len(tf.config.list_physical_devices('GPU')) > 0 else 'CPU'}")

history = model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=[early_stop, model_checkpoint, reduce_lr],
    verbose=1
)

print("\nTraining complete!")

## 5. Evaluate Model Performance

### 5.1 Plot Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# MAE plot
axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Training and Validation MAE')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### 5.2 Make Predictions & Calculate Metrics

In [ ]:
# Make predictions on test set
y_test_pred_scaled = model.predict(X_test_seq, verbose=0)

# Inverse transform predictions and actuals
y_test_pred = scaler_y.inverse_transform(y_test_pred_scaled).flatten()
y_test_actual = scaler_y.inverse_transform(y_test_seq.reshape(-1, 1)).flatten()

# Calculate metrics on log scale
rmse_log = np.sqrt(mean_squared_error(y_test_actual, y_test_pred))
mae_log = mean_absolute_error(y_test_actual, y_test_pred)

# Convert back to original scale (exp(log1p(x)) = x + 1, so use expm1)
y_test_pred_original = np.expm1(y_test_pred)
y_test_actual_original = np.expm1(y_test_actual)

# Calculate metrics on original scale
rmse = np.sqrt(mean_squared_error(y_test_actual_original, y_test_pred_original))
mae = mean_absolute_error(y_test_actual_original, y_test_pred_original)
mape = np.mean(np.abs((y_test_actual_original - y_test_pred_original) / (y_test_actual_original + 1e-10))) * 100

print("\n=== Model Performance ===")
print(f"Log Scale:")
print(f"  RMSE: {rmse_log:.4f}")
print(f"  MAE:  {mae_log:.4f}")
print(f"\nOriginal Scale:")
print(f"  RMSE: {rmse:.2f}")
print(f"  MAE:  {mae:.2f}")
print(f"  MAPE: {mape:.2f}%")

### 5.3 Visualize Predictions vs Actuals

In [ ]:
# Plot predictions vs actuals (sample first 1000 points for visibility)
n_samples = min(1000, len(y_test_actual_original))

plt.figure(figsize=(15, 6))
plt.plot(y_test_actual_original[:n_samples], label='Actual', alpha=0.7, linewidth=1)
plt.plot(y_test_pred_original[:n_samples], label='Predicted', alpha=0.7, linewidth=1)
plt.xlabel('Time Step')
plt.ylabel('Meter Reading')
plt.title('LSTM Predictions vs Actual Energy Consumption (Test Set)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: Predicted vs Actual
plt.figure(figsize=(8, 8))
plt.scatter(y_test_actual_original, y_test_pred_original, alpha=0.3, s=1)
plt.plot([y_test_actual_original.min(), y_test_actual_original.max()],
         [y_test_actual_original.min(), y_test_actual_original.max()],
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Meter Reading')
plt.ylabel('Predicted Meter Reading')
plt.title('LSTM: Predicted vs Actual')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Future Forecasting

Generate predictions for future timesteps using rolling forecast.

In [ ]:
def forecast_future(model, last_sequence, n_steps, scaler_X, scaler_y):
    """
    Forecast future values using rolling prediction.
    
    Parameters:
    - model: trained LSTM model
    - last_sequence: last known sequence (timesteps, n_features)
    - n_steps: number of future steps to predict
    - scaler_X: fitted MinMaxScaler for features
    - scaler_y: fitted MinMaxScaler for target
    
    Returns:
    - predictions: array of predicted values (n_steps,)
    """
    predictions = []
    current_seq = last_sequence.copy()
    
    for _ in range(n_steps):
        # Predict next value
        pred_scaled = model.predict(current_seq[np.newaxis, :, :], verbose=0)[0, 0]
        pred = scaler_y.inverse_transform([[pred_scaled]])[0, 0]
        predictions.append(pred)
        
        # Update sequence (roll forward, assume features stay similar for demo)
        # In production, you'd update features based on known future values
        current_seq = np.roll(current_seq, -1, axis=0)
        # For simplicity, keep last feature values (in practice, update with known future features)
    
    return np.array(predictions)

# Forecast next 168 hours (7 days)
FORECAST_STEPS = 168
last_seq = X_test_seq[-1]

future_predictions_log = forecast_future(model, last_seq, FORECAST_STEPS, scaler_X, scaler_y)
future_predictions = np.expm1(future_predictions_log)

print(f"\nGenerated {FORECAST_STEPS}-step forecast (7 days ahead)")
print(f"Forecast range: {future_predictions.min():.2f} to {future_predictions.max():.2f}")

In [ ]:
# Plot future forecast
plt.figure(figsize=(15, 6))
plt.plot(range(FORECAST_STEPS), future_predictions, label='Forecast', marker='o', markersize=3)
plt.xlabel('Hours Ahead')
plt.ylabel('Predicted Meter Reading')
plt.title('LSTM 7-Day Energy Consumption Forecast')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Save Model & Scalers

Save the trained model and scalers for later use in production.

In [ ]:
import joblib

# Save model (already saved via checkpoint, but we can save again)
model.save(os.path.join(MODELS_DIR, 'lstm_energy_forecaster_final.keras'))

# Save scalers
joblib.dump(scaler_X, os.path.join(MODELS_DIR, 'scaler_X_lstm.pkl'))
joblib.dump(scaler_y, os.path.join(MODELS_DIR, 'scaler_y_lstm.pkl'))

# Save model metadata
metadata = {
    'timesteps': TIMESTEPS,
    'features': feature_cols,
    'rmse': rmse,
    'mae': mae,
    'mape': mape,
    'model_architecture': 'LSTM (128, 64 units)',
    'framework': 'TensorFlow/Keras'
}

import json
with open(os.path.join(MODELS_DIR, 'lstm_model_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print("\nModel, scalers, and metadata saved successfully!")
print(f"Model saved to: {os.path.join(MODELS_DIR, 'lstm_energy_forecaster_final.keras')}")

---

## Summary & Key Insights

### LSTM Model Performance
- **Architecture**: Stacked LSTM (128, 64 units) with dropout regularization
- **Training**: Early stopping, learning rate scheduling, batch size 256
- **Metrics**: RMSE, MAE, MAPE on both log and original scales

### Advantages of LSTM for Energy Forecasting
1. **Temporal Dependencies**: LSTM captures long-term patterns in energy usage
2. **Sequential Learning**: Learns from past 24-hour windows to predict next hour
3. **Non-linear Patterns**: Handles complex relationships between weather, time, and consumption
4. **Scalable**: Can forecast multiple steps ahead with rolling predictions

### Next Steps
1. **Hyperparameter Tuning**: Optimize LSTM units, dropout, learning rate
2. **Feature Engineering**: Add more temporal features (holidays, events)
3. **Ensemble Methods**: Combine LSTM with XGBoost/LightGBM for hybrid model
4. **Production Deployment**: Integrate model into FastAPI for real-time predictions
5. **Attention Mechanisms**: Explore Transformer-based models for better long-range forecasting